# Lab 2.3: Three Explanations of One Risk Score

Run the cells from top to bottom. You do not need to change any code; the last section invites optional changes.

**Data.** A *synthetic* welfare-style dataset generated in this notebook (6,000 invented residents, seed 0). No real person or agency is represented. The generator is deliberately simple and its assumptions are printed below, so that you know the truth the explanations should recover. This is what a synthetic data case buys you: you can check an explanation against a known model and a known data-generating process, which is impossible with a real agency's system.

**What you will do.** Train a risk model, flag the top 10 percent of scores, pick one flagged resident by a fixed rule, and give her three explanations: (1) the model's global importance list, (2) a local surrogate in the style of LIME (Chapter 7), and (3) exact Shapley values (Chapter 8). Then test each explanation for faithfulness and stability, and think about usefulness, which no notebook can measure for you.

In [1]:
import itertools
from math import factorial
import numpy as np
import sklearn
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
print("numpy", np.__version__, "| scikit-learn", sklearn.__version__)

FEATURES = ["age", "months_on_benefit", "household_size", "shared_address",
            "income_changes_12m", "bank_balance_jump", "language_req_met", "caseworker_concern"]

numpy 2.2.6 | scikit-learn 1.7.2


## 1. Generate the synthetic data

Read the generator before you run it. Three design choices matter later:

- The unobserved "true" irregularity depends on discrepancies (a shared address while registered as living alone, a sudden bank balance jump, frequent income changes), on months on benefit and slightly on age. It does **not** depend on language.
- `caseworker_concern` is a subjective flag: partly signal, partly bias against residents who have not met a language requirement.
- The historical label records an irregularity only if someone was investigated, and past investigations targeted residents who had not met the language requirement. So the labels themselves are biased.

In [2]:
def make_data(n=6000, seed=0):
    r = np.random.default_rng(seed)
    age = r.integers(18, 68, n)
    months = np.clip(r.exponential(30, n), 1, 120).round()
    hh = r.choice([1, 2, 3, 4, 5, 6], n, p=[.35, .25, .18, .12, .07, .03])
    shared = (r.random(n) < 0.12).astype(int)
    inc = r.poisson(1.0, n)
    bank = (r.random(n) < 0.06).astype(int)
    lang = (r.random(n) < 0.8).astype(int)
    z = -3.2 + 1.6 * shared + 1.8 * bank + 0.35 * inc + 0.012 * months - 0.01 * (age - 40)
    true_irreg = r.random(n) < 1 / (1 + np.exp(-z))
    concern = (r.random(n) < 0.08 + 0.15 * true_irreg + 0.12 * (1 - lang)).astype(int)
    investigated = r.random(n) < np.where(lang == 0, 0.95, 0.4)
    y = (true_irreg & investigated).astype(int)
    X = np.column_stack([age, months, hh, shared, inc, bank, lang, concern]).astype(float)
    return X, y

X, y = make_data()
print("residents:", len(y), "| share with a recorded irregularity:", round(y.mean(), 3))

residents: 6000 | share with a recorded irregularity: 0.062


## 2. Train the risk model and flag the top 10 percent

*Checkpoint 1:* the test AUC should be about 0.72, and 180 of 1,800 test residents are flagged.

In [3]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
model = GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.05,
                                   random_state=0).fit(Xtr, ytr)
def score(X_):
    return model.predict_proba(X_)[:, 1]
s = score(Xte)
print("test AUC:", round(roc_auc_score(yte, s), 3))
thr = np.quantile(s, 0.9)
flagged = np.where(s >= thr)[0]
print("flag threshold:", round(thr, 3), "| flagged:", len(flagged), "of", len(Xte))

test AUC: 0.724
flag threshold: 0.13 | flagged: 180 of 1800


## 3. Choose one resident by a fixed rule

The rule: the flagged resident whose score is closest to the median flagged score. The chapter calls her Mariam (an invented name for an invented record).

*Checkpoint 2:* Mariam is 45, has been on benefit for 11 months, has two income changes, does **not** share an address, has met the language requirement, and has a caseworker concern flag. Her score is about 0.19.

In [4]:
case = flagged[np.argmin(np.abs(s[flagged] - np.median(s[flagged])))]
x = Xte[case]
for f, v in zip(FEATURES, x):
    print(f"{f:20s} {v:g}")
print("score:", round(s[case], 3))

age                  45
months_on_benefit    11
household_size       2
shared_address       0
income_changes_12m   2
bank_balance_jump    0
language_req_met     1
caseworker_concern   1
score: 0.191


## 4. Explanation 1: the global importance list

Permutation importance measures how much the model's test AUC drops when one feature's values are shuffled across residents (Chapter 6). It describes the model as a whole. Many organizations hand this list to everyone as "the factors the system uses".

*Checkpoint 3:* the top three are caseworker concern, shared address and months on benefit.

In [5]:
pi = permutation_importance(model, Xte, yte, scoring="roc_auc", n_repeats=10, random_state=0)
order_global = list(np.argsort(-pi.importances_mean))
for i in order_global:
    print(f"{FEATURES[i]:20s} {pi.importances_mean[i]:.3f}")

caseworker_concern   0.037
shared_address       0.034
months_on_benefit    0.032
bank_balance_jump    0.028
language_req_met     0.018
income_changes_12m   0.016
age                  0.008
household_size       -0.001


## 5. Explanation 2: a local surrogate (LIME style)

The surrogate switches features on and off at random. A switched-off feature takes the value of a random background resident. It then fits a weighted linear model of the score on the on/off pattern; samples that keep more of Mariam's own values get more weight. The coefficients are the explanation. Chapter 7 covers LIME and its choices in full.

In [6]:
rng = np.random.default_rng(1)
bg = Xtr[rng.choice(len(Xtr), 100, replace=False)]      # 100 background residents

def surrogate(x, background, n_samples=150, seed=0, kernel_width=0.75):
    r = np.random.default_rng(seed); d = len(x)
    M = (r.random((n_samples, d)) < 0.5).astype(float)
    B = background[r.integers(0, len(background), n_samples)]
    f = score(np.where(M == 1, x, B))
    w = np.exp(-((1 - M.mean(1)) ** 2) / kernel_width ** 2)
    A = np.column_stack([np.ones(n_samples), M]) * np.sqrt(w)[:, None]
    return np.linalg.lstsq(A, f * np.sqrt(w), rcond=None)[0][1:]

sur = surrogate(x, bg, seed=0)
for i in np.argsort(-np.abs(sur)):
    print(f"{FEATURES[i]:20s} {sur[i]:+.3f}")

caseworker_concern   +0.130
months_on_benefit    +0.055
shared_address       -0.034
income_changes_12m   +0.017
household_size       +0.014
language_req_met     +0.002
bank_balance_jump    +0.000
age                  +0.000


## 6. Explanation 3: exact Shapley values

With eight features there are only 256 coalitions, so the Shapley values can be computed exactly: for every subset of features, set those features to Mariam's values and the others to the background residents' values, and average the score. Chapter 8 explains the method and its assumptions; Appendix B gives the game theory.

*Checkpoint 4:* caseworker concern (about +0.105) and months on benefit (about +0.061) raise her score most; shared address lowers it (about -0.023). The values add up to her score minus the background mean.

In [7]:
def shapley(x, background):
    d = len(x); phi = np.zeros(d); cache = {}
    def v(S):
        if S not in cache:
            Z = background.copy(); idx = list(S)
            if idx:
                Z[:, idx] = x[idx]
            cache[S] = score(Z).mean()
        return cache[S]
    for i in range(d):
        others = [j for j in range(d) if j != i]
        for k in range(d):
            w = factorial(k) * factorial(d - k - 1) / factorial(d)
            for S in itertools.combinations(others, k):
                phi[i] += w * (v(tuple(sorted(S + (i,)))) - v(S))
    return phi, v(()), v(tuple(range(d)))

phi, base, full = shapley(x, bg)
for i in np.argsort(-np.abs(phi)):
    print(f"{FEATURES[i]:20s} {phi[i]:+.3f}")
print("background mean", round(base, 3), "+ sum", round(phi.sum(), 3), "=", round(full, 3))

caseworker_concern   +0.105
months_on_benefit    +0.061
shared_address       -0.023
income_changes_12m   +0.015
bank_balance_jump    -0.012
language_req_met     -0.006
household_size       -0.005
age                  -0.002
background mean 0.058 + sum 0.133 = 0.191


## 7. The foil changes the answer

"Why was Mariam flagged?" has no single answer until you say *rather than whom*. Compare the background (an average resident) with the most similar unflagged resident.

*Checkpoint 5:* against the neighbor, months on benefit is almost the whole story (about +0.107), and caseworker concern drops to zero because the neighbor has the same flag.

In [8]:
sd = Xtr.std(0); unf = np.where(s < thr)[0]
nb = Xte[unf[np.argmin((((Xte[unf] - x) / sd) ** 2).sum(1))]]
print("neighbor:", dict(zip(FEATURES, nb)), "score", round(score(nb[None])[0], 3))
phi_nb, _, _ = shapley(x, nb[None])
for i in np.argsort(-np.abs(phi_nb)):
    print(f"{FEATURES[i]:20s} {phi_nb[i]:+.3f}")

neighbor: {'age': np.float64(48.0), 'months_on_benefit': np.float64(2.0), 'household_size': np.float64(3.0), 'shared_address': np.float64(0.0), 'income_changes_12m': np.float64(2.0), 'bank_balance_jump': np.float64(0.0), 'language_req_met': np.float64(1.0), 'caseworker_concern': np.float64(1.0)} score 0.103
months_on_benefit    +0.107
household_size       -0.020
age                  +0.000
shared_address       +0.000
income_changes_12m   +0.000
bank_balance_jump    +0.000
language_req_met     +0.000
caseworker_concern   +0.000


## 8. Test 1: faithfulness by deletion

If an explanation says "these are the reasons", removing those reasons (replacing them with typical values) should lower the score quickly. We run this for all 180 flagged residents and average.

A caution you will meet again in Chapter 20: this test replaces features with background values, which is exactly how the Shapley values were computed, so the test is friendly to Shapley. A different replacement scheme could rank the methods differently.

*Checkpoint 6:* after removing two reasons, the Shapley and surrogate orders bring the mean score to about 0.045; the global list leaves it at about 0.17, above the flag threshold.

In [9]:
def deletion(x, order, background):
    out = []
    for k in range(len(order) + 1):
        Z = background.copy(); keep = [j for j in range(len(x)) if j not in order[:k]]
        Z[:, keep] = x[keep]
        out.append(score(Z).mean())
    return np.array(out)

curves = {"shapley": [], "surrogate": [], "global": []}
global_top_raises = 0
for i in flagged:
    xi = Xte[i]
    p, _, _ = shapley(xi, bg)
    curves["shapley"].append(deletion(xi, list(np.argsort(-p)), bg))
    curves["surrogate"].append(deletion(xi, list(np.argsort(-surrogate(xi, bg))), bg))
    curves["global"].append(deletion(xi, order_global, bg))
    global_top_raises += p[order_global[0]] > 0
for k, c in curves.items():
    print(f"{k:10s}", np.round(np.mean(c, axis=0), 3))
print("flagged residents whose score is raised by the global top reason:", global_top_raises, "of", len(flagged))

shapley    [0.26  0.086 0.045 0.035 0.032 0.034 0.039 0.046 0.058]
surrogate  [0.26  0.086 0.046 0.036 0.035 0.036 0.04  0.047 0.058]
global     [0.26  0.219 0.169 0.147 0.083 0.066 0.058 0.059 0.058]
flagged residents whose score is raised by the global top reason: 69 of 180


## 9. Test 2: stability across reruns and backgrounds

The surrogate is random. Rerun it 30 times for Mariam and count how often each feature lands in the top three. Then compare top-3 sets between two runs (Jaccard index: 1 means identical) across all flagged residents, and do the same for Shapley values computed with a different background sample.

*Checkpoint 7:* caseworker concern and months on benefit appear in every run; the third place changes. Mean agreement is about 0.71 for the surrogate with 150 samples, 0.80 with 1,000 samples, and 0.77 for Shapley with a new background.

In [10]:
def topk(v, k=3):
    return set(np.argsort(-np.abs(v))[:k])
def jaccard(a, b):
    return len(a & b) / len(a | b)

freq = np.zeros(8)
for k in range(30):
    for j in topk(surrogate(x, bg, seed=k)):
        freq[j] += 1
for i in np.argsort(-freq):
    print(f"{FEATURES[i]:20s} in top 3 in {freq[i] / 30:.0%} of runs")

bg2 = Xtr[np.random.default_rng(2).choice(len(Xtr), 100, replace=False)]
j150, j1000, jbg = [], [], []
for i in flagged:
    xi = Xte[i]
    j150.append(jaccard(topk(surrogate(xi, bg, seed=1)), topk(surrogate(xi, bg, seed=2))))
    j1000.append(jaccard(topk(surrogate(xi, bg, n_samples=1000, seed=1)), topk(surrogate(xi, bg, n_samples=1000, seed=2))))
    jbg.append(jaccard(topk(shapley(xi, bg)[0]), topk(shapley(xi, bg2)[0])))
print("surrogate 150:", round(np.mean(j150), 2), "| surrogate 1000:", round(np.mean(j1000), 2),
      "| Shapley, new background:", round(np.mean(jbg), 2))

months_on_benefit    in top 3 in 100% of runs
caseworker_concern   in top 3 in 100% of runs
shared_address       in top 3 in 43% of runs
household_size       in top 3 in 13% of runs
income_changes_12m   in top 3 in 13% of runs
bank_balance_jump    in top 3 in 13% of runs
language_req_met     in top 3 in 13% of runs
age                  in top 3 in 3% of runs


surrogate 150: 0.71 | surrogate 1000: 0.8 | Shapley, new background: 0.77


## 10. Test 3: usefulness (your judgment, then a person's)

No code can tell you whether an explanation is useful; that depends on who reads it and what they must do. Answer in your write-up:

1. Mariam asks: "Why me, and what can I do?" Which of the three explanations answers her question? Which of her top reasons can she change, and which can she only contest?
2. Her caseworker asks: "Should I open an investigation?" Which explanation helps that decision, and what would the caseworker need to know about `caseworker_concern` itself?
3. An auditor asks: "Does the system treat residents who have not met the language requirement differently?" Which explanation helps, and what does the global list show about `language_req_met`, a feature that, by construction, has no causal link to irregularity in this data?

## 11. Optional changes

- Change `n_samples` in the surrogate to 50 or 3,000 and rerun Section 9.
- Replace the background with only unflagged residents (`Xtr[score(Xtr) < thr]`) and rerun Section 6. Which foil does that background represent?
- Pick a different resident by changing the rule in Section 3.